In [1]:
%cd /workspace/EBES

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from pathlib import Path
import optuna 
from ebes.pipeline.utils import optuna_df
from optuna.trial import TrialState

/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/workspace/EBES


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#%pip install plotly

In [3]:
def get_run(number, specify="best", rewrite=False):
    path = Path(f"log/{dataset}/{method}/optuna/{number}")
    print(pd.read_csv(path / "results.csv"))
    print((path / "params.txt").read_text())
    save_path = Path(f"configs/specify/{dataset}/{method}")
    save_path.mkdir(parents=True, exist_ok=True)
    save_path = (save_path / f"{specify}.yaml")
    if not rewrite:
        assert not save_path.exists()
    save_path.write_text((path / "params.txt").read_text())

def prepare_data(dataset, method):
    path = Path(f"log/{dataset}/{method}/optuna")
    df, study = optuna_df(path)
    value_pack = ["values_0", "values_1", "values_2", "values_3"]
    col_to_drop = ["datetime_start", "datetime_complete", "system_attrs_fixed_params", "state", *value_pack]
    col_params = [*value_pack, "duration"] + [col for col in df if "params_" in col]
    col_user = [*value_pack, "duration"] + [col for col in df if "user" in col]
    df["duration"] = df["duration"].dt.total_seconds()
    return df, study, col_user, col_params

In [4]:
dataset = "age"
method = "coles"
df, study, col_user, col_params = prepare_data(dataset, method)
print(df.columns)
core_param = "values_0"
print(df.shape, df[~df[core_param].isna()].shape)
test_cols = [col for col in col_user if ("test" in col)]
df[~df[core_param].isna()].sort_values(core_param, ascending=False).iloc[:10, ][[core_param] + test_cols]

Index(['values_0', 'values_1', 'values_2', 'values_3', 'datetime_start',
       'datetime_complete', 'duration',
       'params_data.loaders.unsupervised_train.batch_size',
       'params_model.aggregation.name',
       'params_model.emb_head.params.out_features',
       'params_model.encoder.params.hidden_size',
       'params_model.encoder.params.num_layers',
       'params_model.preprocess.params.cat_emb_dim',
       'params_model.preprocess.params.num_emb_dim',
       'params_model.preprocess.params.num_norm',
       'params_model.preprocess.params.time_process',
       'params_optimizer.params.lr', 'params_optimizer.params.weight_decay',
       'user_attrs_loss_mean', 'user_attrs_loss_std',
       'user_attrs_memory_after_mean', 'user_attrs_memory_after_std',
       'user_attrs_target__age__global__accuracy+f1_macro_mean',
       'user_attrs_target__age__global__accuracy+f1_macro_std',
       'user_attrs_target__anomaly__global__roc_auc+f1_macro+accuracy_mean',
       'user_attrs_

/workspace/EBES/ebes/pipeline/utils.py:168: ExperimentalWarning: JournalStorage is experimental (supported from v3.1.0). The interface can change in the future.
  storage = JournalStorage(JournalFileStorage(f"{path}/study.log"))


,values_0
88,0.493754
87,0.467951
89,0.463799
92,0.456624
104,0.456294
52,0.437556
58,0.434059
51,0.426910
91,0.409454
94,0.403481


In [5]:
#get_run(67, specify="best", rewrite=True)

In [6]:
failed = df[(df["state"] != "COMPLETE") | (df[col_user].isna().any(axis=1))][col_user].index
print(failed)
for fail in df[(df["state"] != "COMPLETE") | (df[col_user].isna().any(axis=1))][col_user].index:
    error_path = Path(f"/home/dev/24/es-bench/log/{dataset}/{method}/optuna/{fail}/ERROR.txt")
    if error_path.exists():
        error = error_path.read_text()
        print(fail, error.split("\n")[-2])
    else:
        print(df.loc[fail])

Index([  1,   9,  29,  31,  34,  43,  48,  54,  56,  59,  60,  61,  64,  66,
        69,  70,  72,  74,  76,  77,  96,  98, 100],
      dtype='int64')
values_0                                                                                     NaN
values_1                                                                                     NaN
values_2                                                                                     NaN
values_3                                                                                     NaN
datetime_start                                                        2026-03-17 22:08:51.080852
datetime_complete                                                     2026-03-17 22:08:55.000409
duration                                                                                3.919557
params_data.loaders.unsupervised_train.batch_size                                            512
params_model.aggregation.name                                            

In [32]:
import pandas as pd
import plotly.express as px
from itertools import combinations

# 1. Подготовка данных (как в прошлом примере)
all_trials = [t for t in study.trials if t.values is not None]
all_values = [t.values for t in all_trials]
trial_numbers = [t.number for t in all_trials]

# Названия колонок
obj_cols = ['Obj 1', 'Obj 2', 'Obj 3', 'Obj 4']
df = pd.DataFrame(all_values, columns=obj_cols)
df['Trial_ID'] = trial_numbers
df['Is_Rank_1'] = [True if num in rank_1_ids else False for num in trial_numbers]
df['Color'] = df['Is_Rank_1'].map({True: 'Rank 1 (Pareto)', False: 'Other Trials'})

# 2. Генерируем 4 комбинации по 3 цели
combos = list(combinations(obj_cols, 3))

for i, combo in enumerate(combos):
    fig = px.scatter_3d(
        df, 
        x=combo[0], y=combo[1], z=combo[2],
        color='Color',
        color_discrete_map={'Rank 1 (Pareto)': 'red', 'Other Trials': 'blue'},
        symbol='Is_Rank_1',
        hover_data=['Trial_ID'] + obj_cols,
        title=f"Комбинация {i+1}: {combo[0]}, {combo[1]}, {combo[2]}",
        opacity=0.7
    )
    
    # Настройка размера точек
    fig.update_traces(marker=dict(size=5))
    fig.show()

In [24]:
optuna.visualization.plot_optimization_history(study, target = lambda t, idx=i: t.values[int(core_param[-1])])

/usr/local/lib/python3.10/dist-packages/optuna/visualization/_utils.py:67: UserWarning:

`target` is specified, but `target_name` is the default value, 'Objective Value'.



In [8]:
import plotly.graph_objects as go
import numpy as np
import optuna

target_names = ["metric_1", "metric_2", "metric_3", "metric_4"]
colors = ["#818CF8", "#FB7185", "#34D399", "#A78BFA"]

trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
trial_numbers = [t.number for t in trials]

fig = go.Figure()

yaxis_refs = ["y", "y2", "y3", "y4"]

# Собираем все значения для нормализации средней кривой
all_values = []
for i in range(len(target_names)):
    values = np.array([t.values[i] for t in trials])
    all_values.append(values)

# Средняя кривая: mean(log(v)) по 4 метрикам, нормализованная в [0,1]
log_mean = np.mean([np.log(np.clip(v, 1e-10, None)) for v in all_values], axis=0)
log_mean_norm = (log_mean - log_mean.min()) / (log_mean.max() - log_mean.min())

# Нормализуем в диапазон первой оси для визуального совмещения
v0 = all_values[0]
log_mean_scaled = log_mean_norm * (v0.max() - v0.min()) + v0.min()

for i, name in enumerate(target_names):
    values = all_values[i]

    fig.add_trace(go.Scatter(
        x=trial_numbers, y=values,
        mode="lines+markers",
        name=name,
        marker=dict(color=colors[i], size=6, opacity=0.6,
                    line=dict(width=0.5, color="white")),
        line=dict(color=colors[i], width=1.5, dash="dot"),
        yaxis=yaxis_refs[i],
        legendgroup=name,
    ))

# Белая средняя кривая — на оси y (первой), шкала не добавляется
fig.add_trace(go.Scatter(
    x=trial_numbers, y=log_mean_scaled,
    mode="lines",
    name="log mean",
    line=dict(color="white", width=2.5),
    yaxis="y",
    opacity=0.85,
))

TICK_STYLE = lambda i, label: dict(
    title=dict(text=label, font=dict(color=colors[i], size=12)),
    tickfont=dict(color=colors[i], size=11),
    showgrid=False,
    zeroline=False,
    linewidth=2,
    linecolor=colors[i],
    ticks="outside",
    ticklen=5,
)

fig.update_layout(
    yaxis=dict(
        side="left",
        **TICK_STYLE(0, target_names[0]),
    ),
    yaxis2=dict(
        side="left",
        overlaying="y",
        anchor="free",
        position=0.0,
        **TICK_STYLE(1, target_names[1]),
    ),
    yaxis3=dict(
        side="right",
        overlaying="y",
        anchor="free",
        position=1.0,
        **TICK_STYLE(2, target_names[2]),
    ),
    yaxis4=dict(
        side="right",
        overlaying="y",
        anchor="free",
        position=0.93,
        **TICK_STYLE(3, target_names[3]),
    ),

    xaxis=dict(
        domain=[0.10, 0.90],
        title=dict(text="Trial number", font=dict(color="#94A3B8", size=13)),
        tickfont=dict(color="#94A3B8"),
        gridcolor="#1E293B",
        zeroline=False,
        showline=True,
        linecolor="#334155",
    ),

    plot_bgcolor="#0F172A",
    paper_bgcolor="#0F172A",

    title=dict(
        text="Optimization History — All Targets",
        font=dict(color="#E2E8F0", size=16),
        x=0.5, xanchor="center",
        y=0.97,
    ),

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.18,
        xanchor="center",
        x=0.5,
        font=dict(color="#CBD5E1", size=12),
        bgcolor="rgba(0,0,0,0)",
        itemclick="toggleothers",
    ),

    height=560,
    margin=dict(l=80, r=80, t=60, b=80),
)

fig.show()

### Params influence

In [33]:
trials = study.trials
trials = [trial for trial in trials if trial.state == TrialState.COMPLETE]
plotted_trials = sorted(trials, key=lambda t: t.value)[:]
plotted_study = optuna.create_study()
for trial in plotted_trials:
    plotted_study.add_trial(trial)

RuntimeError: This attribute is not available during multi-objective optimization.

In [34]:
target = None #lambda t: (t.user_attrs["memory_after_mean"])
target_name = "value"
fig = optuna.visualization.plot_param_importances(plotted_study, target=target, target_name=target_name)
print(fig._data[0]["x"][::-1])
print(fig._data[0]["y"][::-1])
take = 7
params = fig._data[0]["y"][-take:]
not_imp = list(set([col.replace("params_", "") for col in col_params]) - set(params) - {"duration", "value", "system_attrs_fixed_params"})
fig

NameError: name 'plotted_study' is not defined

In [ ]:
params

['model.aggregation.name',
 'model.preprocess.params.time_process',
 'model.preprocess.params.cat_emb_dim',
 'model.preprocess.params.num_emb_dim',
 'model.encoder.params.dropout',
 'optimizer.params.weight_decay',
 'unsupervised_loss.params.margin']

In [35]:
# fig = optuna.visualization.plot_parallel_coordinate(plotted_study, target=target, target_name=target_name, params=['model.encoder.params.pooling', 'pretrain_model.encoder.params.pooling',])
# fig = optuna.visualization.plot_contour(study, target=target, target_name=target_name, params=params+not_imp)
fig = optuna.visualization.plot_slice(study, target=target, target_name=target_name)#, params=["model.encoder.params.num_layers"] )
# fig = optuna.visualization.plot_optimization_history(study, target=target, target_name=target_name, error_bar=False)
# targets = lambda t: (t.user_attrs["memory_after_mean"], t.user_attrs["val_metric_mean"])
# target_names = ["memory_after_mean", "val_metric_mean"]
# fig = optuna.visualization.plot_pareto_front(study, targets=targets, target_names=target_names)
fig

ValueError: If the `study` is being used for multi-objective optimization, please specify the `target`.